In [ ]:
# Import packages
import sys
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


# Local module imports
import microscopy_analysis.d00_utils.utilities as utils
import microscopy_analysis.d00_utils.dirnames as dn
import microscopy_analysis.d04_plot_data.restructure_data as rd
import microscopy_analysis.d04_plot_data.plot_from_dataframe as pfd

import starbars
import warnings

warnings.filterwarnings('ignore')
%matplotlib notebook
%matplotlib inline

In [ ]:
# set default style for graphs
smallpts_fillcolor = '#DBDBDB'
smallpts_edgecolor = '#AFAFAF'
ticks_fontsize = 8
axislabel_fontsize = 10
linewidth = 0.5

rc = {'svg.fonttype':'none', 
      'font.family':'Arial',
      'figure.figsize': (4.5,5),
      'figure.dpi': 150,
      'axes.linewidth': linewidth,
      'axes.labelweight':'bold',
      'axes.labelsize': axislabel_fontsize,
      'axes.labelpad':7.5
     }

sns.set(rc)
sns.set_style("ticks")

ctrl_color = '#dd8452'
deact_color = '#4c72b0'
ctrldeact_palette = [ctrl_color, deact_color]

In [ ]:
df_path = Path(input('Please enter the full path for the dataframe:'))

In [ ]:
df = pd.read_csv(df_path)
df.head()

In [ ]:
# omit cells marked for omission
cmp_df = df
if 'omit' in df.columns:
    num_cells_prefilter = len(cmp_df)
    cmp_df = cmp_df[cmp_df['omit']!='Y']
    num_cells_postfilter = len(cmp_df)
    print(f'num cells omitted: {num_cells_prefilter - num_cells_postfilter}')
else:
    print('No column indicating cells to omit.')

# omit cells marked for omission for actin analysis
actin_df = df
if 'actin omit' in df.columns:
    num_cells_prefilter = len(actin_df)
    actin_df = df[df['actin omit']!='Y']
    num_cells_postfilter = len(actin_df)
    print(f'num cells omitted for actin analysis: {num_cells_prefilter - num_cells_postfilter}')
else:
    print('No column indicating cells to omit for actin analysis.')

In [ ]:
tables_dirpath = df_path.parent
graphs_dirpath = df_path.parent / 'graphs'
graphs_dirpath.mkdir(parents=True, exist_ok=True)

In [ ]:
# Save cell counts for compaction data
cmp_counts = cmp_df.groupby(['condition', 'experiment'])['condition'].count()
cmp_counts_df_path = tables_dirpath / 'cmp_data_counts.csv'
cmp_counts.to_csv(cmp_counts_df_path, index=True)

# Save cell counts for actin data
actin_counts = actin_df.groupby(['condition', 'experiment'])['condition'].count()
actin_counts_df_path = tables_dirpath / 'actin_data_counts.csv'
actin_counts.to_csv(actin_counts_df_path, index=True)

actin_counts

In [ ]:
# Calculate biological replicate averages for compaction data
groupbycols = ['condition', 'experiment']
ycols = ['cell area', 'compacted area', 'CAAX-positive area', '% compaction', 'mean caax int (cell)', 'caax integrated density',
         'mean caax int (compacted)', 'mean caax int (caax)']
agg_cols = {y: 'mean' for y in ycols}
cmp_biorepavg_df = cmp_df.groupby(groupbycols, as_index=False).agg(agg_cols)
cmp_biorepavg_df_path = tables_dirpath / 'cmp_biorep_avgs.csv'
utils.safe_save_csv(cmp_biorepavg_df, cmp_biorepavg_df_path)
cmp_biorepavg_df

# Calculate biological replicate averages for actin data
ycols = ['mean actin int (cell)', 'actin integrated density', 'mean actin int (compacted)', 'mean actin int (caax)']
agg_cols = {y: 'mean' for y in ycols}
actin_biorepavg_df = actin_df.groupby(groupbycols, as_index=False).agg(agg_cols)
actin_biorepavg_df_path = tables_dirpath / 'actin_biorep_avgs.csv'
utils.safe_save_csv(actin_biorepavg_df, actin_biorepavg_df_path)

In [ ]:
def get_stats_and_graph(x, y, x_order, df, avg_df, stats_df, graphs_dirpath, graphname=None, figsize=(2.5,3.25), palette=ctrldeact_palette):
    group1_name = x_order[0]
    group2_name = x_order[1]

    idx = len(stats_df)
    stats_df.loc[idx, 'comparison value'] = y
    stats_df.loc[idx, 'comparison group 1'] = group1_name
    stats_df.loc[idx, 'comparison group 2'] = group2_name
    group1 = avg_df.loc[avg_df['condition']==group1_name, y].values
    group2 = avg_df.loc[avg_df['condition']==group2_name, y].values
    stats_df.loc[idx, 'group 1 n'] = len(group1)
    stats_df.loc[idx, 'group 2 n'] = len(group2)

    # Shapiro-Wilk test to test for normality on the difference between groups
    diff = group2 - group1
    normality_stat, normality_pvalue = stats.shapiro(diff)
    stats_df.loc[idx, 'Shapiro-Wilk normality stat'] = normality_stat
    stats_df.loc[idx, 'Shapiro-Wilk normality p-value'] = normality_pvalue
    
    # paired t-test
    stat, pvalue = stats.ttest_rel(group1, group2)
    stats_df.loc[idx, 'paired t-test stat'] = stat
    stats_df.loc[idx, 'paired t-test p-value'] = pvalue

    # create and save graph
    fig, ax = plt.subplots(figsize=figsize, dpi=150)
    sns.swarmplot(x=x, y=y, data = df, order=x_order, size=2.5, ax=ax, color=smallpts_fillcolor, edgecolor=smallpts_edgecolor, linewidth=0.2)
    sns.swarmplot(x=x, y=y, order=x_order, data = avg_df, hue='condition', palette=palette, legend=False, size=7, ax=ax)
    sns.pointplot(data=avg_df, x=x, y=y, order=x_order, linestyle='', errorbar='se', marker='_', markersize=25, markeredgewidth=2.5, 
                  zorder=3, color='k', capsize=0.1, err_kws={'linewidth':1}, ax=ax)
    starbars.draw_annotation([(group1_name, group2_name, pvalue)], line_width=linewidth, color='k', ax=ax)
    sns.despine()
    plt.ylim(0)
    plt.xlim(-0.7, 1.7)
    ax.xaxis.label.set_visible(False)
    ax.tick_params(direction='out', width=linewidth, labelsize=ticks_fontsize)
    plt.show()

    if graphname is None:
        graphname = y.replace('(', '').replace(')', '').replace(' ', '_').replace('%', 'perc') + '.svg'
    fig.savefig(graphs_dir / graphname, format='svg', bbox_inches='tight')
    
    return stats_df

In [ ]:
stats_df

In [ ]:
stats_df = pd.DataFrame()

x_order = ['Control', 'DeAct']
x = 'condition'

ycols = cmp_biorepavg_df.columns.tolist()[2:]
for y in ycols:
    if 'caax' in y:
        figsize = (3, 3.25)
    else:
        figsize = (2.5, 3.25)
    stats_df = get_stats_and_graph(x, y, x_order, df=cmp_df, avg_df=cmp_biorepavg_df, stats_df=stats_df, graphs_dirpath=graphs_dirpath, figsize=figsize)

ycols = actin_biorepavg_df.columns.tolist()[2:]
for y in ycols:
    figsize = (2.5, 3.25)
    stats_df = get_stats_and_graph(x, y, x_order, df=actin_df, avg_df=actin_biorepavg_df, stats_df=stats_df, graphs_dirpath=graphs_dirpath, figsize=figsize)

utils.safe_save_csv(stats_df, tables_dirpath / 'stats.csv')

In [ ]:
# Reformat graph to enable plotting actin intensity in compacted vs non-compact regions in ctrl cells
actin_df_ctrl = actin_df[actin_df['condition']=='Control']
actin_dfm_ctrl = actin_df_ctrl.melt(id_vars=['UID', 'condition'], value_vars=['mean actin int (caax)', 'mean actin int (compacted)', 'mean actin int (cell)'], value_name='mean actin int', col_level=None, ignore_index=True)

actin_avg_df_ctrl = actin_biorepavg_df[actin_biorepavg_df['condition']=='Control']
actin_avg_dfm_ctrl = actin_avg_df_ctrl.melt(id_vars=['experiment', 'condition'], value_vars=['mean actin int (caax)', 'mean actin int (compacted)', 'mean actin int (cell)'], value_name='mean actin int', col_level=None, ignore_index=True)

regions = ['cell', 'caax', 'compacted']
for reg in regions:
    actin_dfm_ctrl.loc[actin_dfm_ctrl['variable'].str.contains(reg), 'condition'] = reg
    actin_avg_dfm_ctrl.loc[actin_avg_dfm_ctrl['variable'].str.contains(reg), 'condition'] = reg

In [ ]:
x = 'condition'
x_order = ['caax', 'compacted']
y = 'mean actin int'

ctrl_palette = [ctrldeact_palette[0], ctrldeact_palette[0]]
stats_df = get_stats_and_graph(x, y, x_order, df=actin_dfm_ctrl, avg_df=actin_avg_dfm_ctrl, stats_df=stats_df, graphs_dirpath=graphs_dirpath, palette=ctrl_palette)

In [ ]:
cmp_df_ctrl = cmp_df[cmp_df['condition']=='Control']
cond = 'ctrl'
figname = f'mean_caax_int_caax_vs_perc_comp_{cond}'
x_label = 'mean caax int (cell)'
y_label = '% compaction'
# plt.savefig(graphs_dir / f'{figname}.png')
sns.lmplot(x=x_label, y=y_label, data=cmp_df_ctrl, hue='condition', palette=[ctrl_color], legend=False)
sns.lmplot(x=x_label, y=y_label, data=cmp_df_ctrl, hue='experiment', legend=False)
plt.show()

cmp_df_ctrl = cmp_df[cmp_df['condition']=='Control']
cond = 'ctrl'
figname = f'mean_caax_int_caax_vs_perc_comp_{cond}'
x_label = 'mean mRubycaax int (cell)'
y_label = '% compaction'
# plt.savefig(graphs_dir / f'{figname}.png')
sns.lmplot(x=x_label, y=y_label, data=cmp_df_ctrl, hue='condition', palette=[ctrl_color], legend=False)
sns.lmplot(x=x_label, y=y_label, data=cmp_df_ctrl, hue='experiment', legend=False)
plt.show()

# sns.lmplot(x=x_label, y=y_label, data=df_subset, hue='experiment', color='k')
# plt.show()

# slope, intercept, r_value, p_value, std_err = stats.linregress(df_subset[x_label],df_subset[y_label])
# print(f'{cond}: slope: {slope}, intercept: {intercept}, r_value: {r_value}, p_value: {p_value}, std_err: {std_err}')

# figname = f'caax_intden_vs_perc_comp_{cond}'
# df_subset.loc[:, 'caax integrated density'] = df_subset['mean caax int (cell)'] * df_subset['cell area']
# x_label = 'caax integrated density'
# sns.scatterplot(df_subset, x=x_label, y=y_label, legend=False, size=3, color='k')
# plt.savefig(graphs_dir / f'{figname}.png')
# plt.ylim(0, 1)
# plt.show()

actin_df_deact = actin_df[actin_df['condition']=='DeAct']
cond = 'DeAct'
# figname = f'mean_caax_int_cell_vs_perc_comp_{cond}'
# x_label = 'mean caax int (caax)'
# y_label = '% compaction'
#sns.lmplot(x=x_label, y=y_label, data=cmp_df_deact, hue='experiment', palette=[deact_color], legend=False)
sns.lmplot(x=x_label, y=y_label, data=cmp_df_deact, hue='condition', palette=[deact_color], legend=False)
sns.lmplot(x=x_label, y=y_label, data=cmp_df_deact, hue='experiment', legend=True)
sns.lmplot(x='mean caax int (cell)', y='mean actin int (cell)', data=actin_df_deact, legend=True)

# plt.savefig(graphs_dir / f'{figname}.png')
# plt.ylim(0, 1)
# plt.show()
# slope, intercept, r_value, p_value, std_err = stats.linregress(df_subset[x_label],df_subset[y_label])
# print(f'{cond}: slope: {slope}, intercept: {intercept}, r_value: {r_value}, p_value: {p_value}, std_err: {std_err}')

# sns.lmplot(x=x_label, y=y_label, data=df_subset)
# plt.show()

# figname = f'GFPcaax_intden_vs_perc_comp_{cond}'
# df_subset.loc[:, 'caax integrated density'] = df_subset['mean caax int (cell)'] * df_subset['cell area']
# x_label = 'caax integrated density'
# sns.scatterplot(df_subset, x=x_label, y=y_label, legend=False, size=3, color='k')
# plt.savefig(graphs_dir / f'{figname}.png')
# plt.ylim(0, 1)
# plt.show()


In [ ]:

figname = 'mean_actin_int_cell'

actin = 'mean_actin_int_cell'
x_label = 'mean actin int (cell)'
y_label = '% compaction'
plt.ylim(0, 1)
ax = sns.scatterplot(actin_df, x=x_label, y=y_label, hue='condition', legend=True, size=3, color='k')
plt.show()

sns.lmplot(x=x_label, y=y_label, data=actin_df, )
plt.show()

x_label = 'mean actin int (cell)'
y_label = '% compaction'
sns.lmplot(x=x_label, y=y_label, data=actin_df)
plt.show()

x_label = 'mean actin int (cell)'
y_label = '% compaction'
sns.lmplot(x=x_label, y=y_label, data=actin_df[actin_df['tx']=='281'])
plt.show()

df.loc[:, 'actin integrated density'] = df['mean actin int (cell)'] * df['cell area']
x_label = 'actin integrated density'
y_label = '% compaction'
plt.ylim(0, 1)
ax = sns.scatterplot(actin_df, x=x_label, y=y_label, hue='condition', size=3, color='k')
sns.lmplot(x=x_label, y=y_label, data=actin_df, hue='condition')
plt.show()


sns.lmplot(x='cell area', y='% compaction', data=actin_df, hue='condition')
plt.show()

In [ ]:
stats_df = stats_df.drop_duplicates()
stats_df_path = tables_dirpath / 'stats.csv'
utils.safe_save_csv(stats_df, stats_df_path)